In this assignment, you will use a pre-trained convnet to produce features for a classifier that can detect a single object type. This notebook has some code to help you get started. 

In [1]:
import pandas as pd
import os
from os import listdir
from os.path import isfile, join
import os.path as osp
from tqdm import tqdm_notebook as tqdm

In [4]:
pip install imutils

  Preparing metadata (setup.py): started
  Preparing metadata (setup.py): finished with status 'done'
  Created wheel for imutils: filename=imutils-0.5.4-py3-none-any.whl size=25853 sha256=6b4b568bb0e20ac6452f4f2e6d54803fe775cc43c8cc23e255066b7f1b32e42c
  Stored in directory: c:\users\sharadha kasi\appdata\local\pip\cache\wheels\4b\a5\2d\4a070a801d3a3d93f033d3ee9728f470f514826e89952df3ea
Successfully built imutils
Note: you may need to restart the kernel to use updated packages.


In [7]:
pip install opencv-python

   ---------------------------------------- 0.0/38.6 MB ? eta -:--:--
   ---------------------------------------- 0.0/38.6 MB ? eta -:--:--
    --------------------------------------- 0.5/38.6 MB 10.7 MB/s eta 0:00:04
   - -------------------------------------- 1.6/38.6 MB 16.7 MB/s eta 0:00:03
   --- ------------------------------------ 3.3/38.6 MB 23.3 MB/s eta 0:00:02
   --- ------------------------------------ 3.5/38.6 MB 17.1 MB/s eta 0:00:03
   --- ------------------------------------ 3.7/38.6 MB 15.7 MB/s eta 0:00:03
   ---- ----------------------------------- 3.9/38.6 MB 13.9 MB/s eta 0:00:03
   ---- ----------------------------------- 4.2/38.6 MB 12.7 MB/s eta 0:00:03
   ---- ----------------------------------- 4.4/38.6 MB 11.8 MB/s eta 0:00:03
   ---- ----------------------------------- 4.7/38.6 MB 11.1 MB/s eta 0:00:04
   ----- ---------------------------------- 4.9/38.6 MB 10.5 MB/s eta 0:00:04
   ----- ---------------------------------- 5.2/38.6 MB 10.1 MB/s eta 0:00:04
  

In [9]:
pip install google_images_download


  Preparing metadata (setup.py): started
  Preparing metadata (setup.py): finished with status 'done'
   ---------------------------------------- 0.0/10.0 MB ? eta -:--:--
   - -------------------------------------- 0.3/10.0 MB 5.9 MB/s eta 0:00:02
   ---- ----------------------------------- 1.1/10.0 MB 11.7 MB/s eta 0:00:01
   -------- ------------------------------- 2.2/10.0 MB 15.7 MB/s eta 0:00:01
   ------------- -------------------------- 3.4/10.0 MB 19.9 MB/s eta 0:00:01
   --------------- ------------------------ 3.8/10.0 MB 16.1 MB/s eta 0:00:01
   --------------- ------------------------ 4.0/10.0 MB 14.8 MB/s eta 0:00:01
   ---------------- ----------------------- 4.2/10.0 MB 12.7 MB/s eta 0:00:01
   ----------------- ---------------------- 4.5/10.0 MB 11.9 MB/s eta 0:00:01
   ------------------ --------------------- 4.7/10.0 MB 11.2 MB/s eta 0:00:01
   ------------------- -------------------- 5.0/10.0 MB 10.6 MB/s eta 0:00:01
   -------------------- ------------------- 5.1/

In [13]:
import imutils
from imutils import paths
import requests
#import cv2
from google_images_download import google_images_download

img_folder = 'downloads'

In [14]:
def build_arguments(word):
    args = {}
    args['keywords'] = word
    args['limit'] = 100
    args['format'] = 'png'
    args['usage_rights'] = 'labeled-for-nocommercial-reuse'
    return args

response = google_images_download.googleimagesdownload()

### Gather positive examples

Pick a word. For example, "red" or "santa" or "horse". 

Now you will need to find "positive" image examples of that word. For example, if you chose "red" as your word, you will need to find images of red things. You are free to use Google Image search or something similar. File types shouldn't matter, but try to stick with .png and .jpg files.

You'll need at least 100 positive example images. Put them in the folder called `pos`. 

### Gather negative examples

Now you need to think about negative examples; i.e., things that are *not* examples of your word. You can either just find random images, or look for specific negative examples. For example, if you chose the word "red" then it might work best if you find negative examples that are other colors, especially colors close to red. 

You'll need at least 200 negative example images. Put them in the folder called `neg`. 

## 1.) Run the following cell

* This imports needed Keras libraries
* Then, it gets the trained VGG19 imagenet model
* Then, it prints out the names of all the layers in that model

In [2]:
import numpy as np
from tensorflow.keras.applications import VGG19
from tensorflow.keras.applications.vgg19 import preprocess_input
from tensorflow.keras.preprocessing import image
from tensorflow.keras.models import Model

base_model = VGG19(weights='imagenet',include_top=True)
xs,ys=224,224

for layer in base_model.layers:
    print(layer.name)

input_1
block1_conv1
block1_conv2
block1_pool
block2_conv1
block2_conv2
block2_pool
block3_conv1
block3_conv2
block3_conv3
block3_conv4
block3_pool
block4_conv1
block4_conv2
block4_conv3
block4_conv4
block4_pool
block5_conv1
block5_conv2
block5_conv3
block5_conv4
block5_pool
flatten
fc1
fc2
predictions


### 2.) Determine your output layer

- try `predictions` first
- note the layers printed out above; you can use any of those laters
- pay attention to output shape of each layer! predictions is a vector of size 1000, for example

In [3]:
layer = 'predictions'

model = Model(inputs=base_model.input, outputs=base_model.get_layer(layer).output)

### Run the following cell

- These functions are to help you perform transfer learning

In [ ]:
def get_image(img_path, xs,ys):
    x = image.load_img(img_path, target_size=(xs, ys))
    x = image.img_to_array(x)
    x = np.expand_dims(x, axis=0)
    return x

def get_img_features(model, img):
    img = preprocess_input(img)
    yhat = model.predict(img)
    return yhat

def get_image_features(word):
    files = [f for f in listdir(word)] # grab all of the images in the folder
    image_vectors = []
    for f in tqdm(files):
        img = get_image(osp.join(word, f), xs, ys) 
        x_feats = get_img_features(model, img).flatten() # get features for each image
        image_vectors.append(x_feats) 
    return np.array(image_vectors)

## 3.) Evaluate a classifier for your `word`

* Using the positive and negative output from `model`, train a classifier (it can be a linear classifier from scikit-learn, if you'd like, but I would recommend the Keras Dense network we built for the previous assignment). 
* You'll need to split your data into Train and Test (I would recommend using half of the data for training, half for testing; you may opt for downloading more positive and negative examples)
* your classifier can be any scikit classifier, but you can also use a neural network of some kind

In [ ]:
pos_images = get_image_features('pos') # get positive image vectors
neg_images = get_image_features('neg') # get negative image vectors

### Prepare the data. Split to train/test sets

### Define model, train

### Evaluate

### 4.) Use CLIP

* Repeat steps 3 and 4 above, only this time using the [CLIP](https://github.com/openai/CLIP) model
  
  To get image features, use the following example: `image = preprocess(Image.open("CLIP.png")).unsqueeze(0).to(device)`

(see also the last code section of the README for the CLIP github repo on training a classifier using CLIP features)
  
  
* (Answer in a markdown cell): Which model+layer works the best for this data? Why do you think that is?
* What makes for good positive examples? What makes for good negative examples? Why does the choice of negative examples matter?